# 자습 09. 로컬 VLA 개발 프로젝트 설계

> **선택 자습 — 본수업 8시간에 포함하지 않습니다.** 코드 셀은 소형 합성 예제이며 GPU 학습·다운로드·로봇 구동을 자동 실행하지 않습니다. 실제 작업 명령은 설명의 터미널 예시로 제공하며 준비물을 먼저 갖춥니다. TODO는 직접 작성하고 아래 정답·해설로 검증하세요. [전체 자습 경로](selfstudy/README.md)


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| 계약 항목 누락·검사 실패 | task·성공 조건·카메라·상태·행동·장비·정지 기준을 실제 값으로 채웁니다. | 단위·좌표계를 포함한 계약 설명 가능 |
| `Exec format error`, CUDA 인식 실패 | ARM/x86와 Python 경로·PyTorch/드라이버 환경을 확인합니다. Jetson 이미지를 PC용으로 간주하지 않습니다. | 선택 환경에서 CUDA 인식 후 작은 계산 성공 |
| 모델·데이터 경로 없음 | 자습용 실제 시연과 기본 모델은 별도 준비물입니다. 전체 자습 안내의 경로와 준비 단계를 확인합니다. | 실제 파일과 계약 확인 |

**예시에 없는 오류를 확인할 곳:** [VLA 단계별 오류 사전](selfstudy/TROUBLESHOOTING.md)의 09 절과 [준비 안내](selfstudy/README.md). 훈련은 터미널 stdout/stderr와 checkpoint 설정, 재생은 지정 output의 결과 파일을 봅니다.

VLA 훈련·저장 관측 재생에는 noVNC 창이 열리지 않습니다. 본수업 GUI rollout을 다시 실행한 경우에만 FINISHED 후 MuJoCo 창을 닫습니다.

[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 여기서 무엇을 하나요?

01부터 08까지는 이미 학습된 **LIBERO용 정책** 을 실행했습니다. 이제 엔지니어가 자신의 과제와 장비에 맞는 정책을 만드는 개발 과정을 설계합니다. 이 자습의 학습은 주로 **기존 SmolVLA의 미세조정** 을 뜻합니다. 거대한 사전학습을 처음부터 재현하는 과정과 다릅니다.

**완료 결과:** 작업 성공 정의, 입력·행동 계약, 학습/검증/시험 분리 원칙, 장비별 실행 계획을 한 장에 작성합니다. 예상 읽기·설계 시간 60분이며 실제 데이터 수집·훈련 시간은 별도입니다.

![VLA 개발 전체 흐름](../교재그림/vla-selfstudy-lifecycle.svg)

관측 → 정책 → 행동이라는 추론 경로 앞에 시연 데이터와 학습이 붙고, 뒤에 평가와 배포 관리가 붙습니다. 실패 사례는 원인을 분류한 뒤 다음 데이터 수집으로 돌아갑니다. 같은 시험 데이터를 반복해서 보며 설정을 맞추면 시험 세트도 개발 데이터가 되므로 마지막 확인용 세트를 따로 남깁니다.


## 두 경로를 섞지 않기

<!-- procedure:lesson-45-8873ef:start -->
![내 장비의 VLA 개발 계약](../교재그림/절차/lesson-45.svg)

09–13은 보유 하드웨어로 개발하는 선택 자습입니다. 본수업 모델을 그대로 실물 장비에 연결하지 않습니다.
<!-- procedure:lesson-45-8873ef:end -->


| 경로 | 시작점 | 추가로 필요한 것 | 완료 판단 |
|---|---|---|---|
| 시뮬레이션 확장 | 본수업 LIBERO 관측·OSC 행동 | 시연 수집 어댑터, 시간별 전체 관측과 실제 적용 행동 | 보지 않은 초기 조건의 rollout |
| 내 로봇 적용 | 자신의 로봇·카메라·제어기 | 보정, 조작 장치, 시연, 입출력 어댑터 | 제한된 실제 작업에서 반복 성공과 중단 동작 |

본수업의 상태8·행동7은 LIBERO 계약입니다. 관절6개 로봇의 관절 목표값과 같은 의미가 아닙니다. `smolvla_libero`를 새로운 로봇의 기본 모델로 간주하지 마세요. 새로운 로봇 경로에서는 준비한 `smolvla_base` 계열 체크포인트와 자신의 시연으로 미세조정하는 구성을 검토합니다. 모델의 지원 입력·행동 크기와 전처리도 함께 확인합니다.

07의 `initial-observation.npz`와 `action-chunks.npy`만으로는 시연 데이터가 완성되지 않습니다. 앞의 파일에는 시작 관측만 있고, 뒤에는 예측했지만 적용하지 않은 행동까지 들어갑니다. 학습 예제는 **각 시점의 관측과 그때 실제로 적용한 행동** 이 연결되어야 합니다. 정책이 만든 궤적은 사람 전문가 시연과 구분해 출처와 성공 여부를 기록합니다.


## 로컬 장비를 역할별로 배치하기

여기서 로컬은 소유한 장비에서 데이터를 처리하고 학습하는 구성이며, Jetson 한 대에 모든 작업을 동시에 올린다는 뜻은 아닙니다.

| 장비 | 먼저 맡길 작업 | 실측할 항목 |
|---|---|---|
| Jetson | 수집, 센서 연결, 입력 검사, 배포 추론 | 공유 RAM, 온도, 카메라 주기, 추론 지연 |
| 별도 로컬 NVIDIA GPU PC | 미세조정·반복 평가 | VRAM 최대치, step 시간, 저장 여유 공간 |
| Jetson만 있는 경우 | batch 1의 짧은 forward/backward 시범부터 | OOM 여부, 메모리 여유, 지속 속도·발열 |

Jetson에서 추론이 된다는 사실만으로 훈련 가능 여부를 결론 내릴 수 없습니다. 역전파 활성값·gradient·optimizer state가 추가됩니다. 특정 RAM 용량이면 된다고 보장하지 않고 자신의 구성에서 측정합니다. 같은 Jetson의 두 수강생은 본수업 추론과 자습 훈련을 겹치지 않게 계획합니다.

**환경 분리:** 본수업 이미지는 추론 검증용이고 ARM64 Jetson 의존성 패치가 있습니다. x86 GPU PC로 이 이미지를 그대로 복사해 실행하지 않습니다. PC에서는 별도 환경에 LeRobot 버전·PyTorch/CUDA 호환 조합을 고정하고 11의 준비 검사부터 진행합니다. 새로운 API 문서를 따라 수업 컨테이너를 무조건 업그레이드하지 않습니다.


## TODO 1 — 개발 계약 작성

아래 딕셔너리의 빈 항목을 **자신의 실제 장비 기준** 으로 채우세요. 아직 장비가 없으면 ‘가정’임을 써 둡니다. `success`에는 이미지상 인상 대신 측정 가능한 조건을, `stop`에는 정책과 독립된 중단 수단을 적습니다.


In [ ]:
project = {
    'task': '',
    'success': '',
    'cameras': '',  # 이름·위치·해상도·RGB 방향
    'state': '',    # 성분 순서·단위·좌표계
    'action': '',   # 제어 모드·성분 순서·단위·목표 주기
    'train_device': '',
    'deployment_device': '',
    'held_out_conditions': '',
    'stop': '',
}
missing = [key for key, value in project.items() if not value.strip()]
print('TODO 미완료 항목:', missing)

### 정답 예시와 해설

유일한 장비 정답은 없습니다. 예를 들어 ‘고정된 작업대의 파란 블록을 트레이 안에 놓고 그리퍼를 연 후 2초간 유지’를 성공 정의로 정할 수 있습니다. 카메라는 `front`·`wrist`, 상태는 보정된 관절 위치, 행동은 같은 관절 순서의 목표 위치로 정의하되 **단위는 장치 API에서 확인** 합니다. 학습은 로컬 GPU PC, 추론은 Jetson, 시험 조건은 학습에 쓰지 않은 수집 날짜·배치로 나눕니다.

‘잘 집는다’만 쓰면 평가자마다 성공 판정이 달라집니다. ‘숫자 6개’만 쓰면 rad와 degree 또는 목표 위치와 속도가 뒤섞입니다. 중단 방식은 제어기 watchdog과 작업자가 사용할 독립 정지 수단을 실제 장비에서 확인해야 합니다. `action=[0,...]`가 정지라고 가정하면 절대 위치 제어에서는 원점으로 이동할 수 있습니다.

**TODO 2 — 설계 검토:** 카메라를 90도 돌렸을 때, 제어 주기를 바꿨을 때, 학습 때와 다른 그리퍼를 장착했을 때 무엇을 재검증해야 하는지 적으세요.

**해설:** 카메라는 입력 분포·보정·좌우 방향, 주기는 행동 간격·묶음 실행 시간·시연 시간 정렬, 그리퍼는 상태·명령 단위·가동 범위가 변합니다. 단순 파일 교체로 해결된다고 보지 않고 데이터·어댑터·평가를 함께 갱신합니다.


## 다음 단계와 근거

프로젝트 계약을 확정한 뒤 [10 시연 데이터](10_selfstudy_demonstration_data.ipynb)로 진행합니다. 로컬 폴더 구조와 준비 목록은 [자습 안내](selfstudy/README.md)를 봅니다.

SmolVLA의 관측·언어·행동 묶음 구성과 미세조정 개요: [공식 SmolVLA 문서](https://huggingface.co/docs/lerobot/v0.4.3/en/smolvla). 본수업의 실제 연결은 [course_runtime.py](scripts/course_runtime.py)의 `make_batch`와 `make_env`가 기준입니다. 문서 버전과 현재 설치 버전 0.4.4의 차이는 실제 소스로 확인합니다.
